# 04 – Datavoorbereiding: van ruwe data naar de modeltabel (zonder grafieken)

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | Alle cleaning- en voorbereidingsstappen die in `02_eda_cleaning` en `03_hypotheses` nodig bleken, achter elkaar en zonder grafieken. Dit notebook (samen met `01_download`) is de volledige weg van ruwe zip's naar de tabel waarop alle modellen trainen. |
| **Input** | `data/station_hour.parquet`, `data/stations.parquet`, `data/weather_2024.parquet` (uit `01_download`) |
| **Output** | `data/model_table.parquet`, `data/zones.json`, `data/station_zones.parquet` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

## Stappen (en waar ze onderbouwd worden)
| Stap | Wat | Waarom (zie) |
|---|---|---|
| 1 | Alleen vertrekken met starttijd, startstation en coördinaten in NYC (al in 01) | 02 §2 |
| 2 | Station-id's normaliseren `3423.1` → `3423.10` (al in 01) | 02 §3 |
| 3 | Niet-publieke stations (depots, laadkades, testlocaties) en Jersey City/Hoboken weglaten | 02 §5 |
| 4 | Stations groeperen tot 30 zones (KMeans op locatie, gewogen met ritten) | 02 §8 |
| 5 | Volledig rooster zone × uur, uren zonder vertrek = 0 | 02 §10 |
| 6 | Weer per uur erbij (Open-Meteo) | 03 |
| 7 | Vaste split: laatste 7 dagen van elke maand = test | hieronder |

**Wat we bewust níet doen:** ritten filteren op duur. Een rit van 5 uur is nog altijd een vertrek (02 §4). En we vervangen geen "uitschieters" in het aantal vertrekken: drukke uren zijn echte vraag, geen meetfouten.

In [1]:
import json

import pandas as pd

import citibike as cb

station_hour = pd.read_parquet(cb.DATA_DIR / "station_hour.parquet")
stations = pd.read_parquet(cb.DATA_DIR / "stations.parquet")
weather = pd.read_parquet(cb.DATA_DIR / f"weather_{cb.YEAR}.parquet")
print(f"{len(station_hour):,} station-uren, {len(stations):,} stations, {len(weather):,} weer-uren")

9,535,664 station-uren, 2,280 stations, 8,784 weer-uren


## Stappen 3–7
`prepare_model_table` in `citibike.py` voert de stappen uit. De KMeans gebruikt een vaste `random_state`, zodat de zones bij elke uitvoering identiek zijn.

In [2]:
table, zones, public_stations = cb.prepare_model_table(station_hour, stations, weather)
dropped = cb.clean_stations(stations).query("not keep")
print(f"Weggelaten stations: {len(dropped)} ({dropped['trips'].sum():,} vertrekken = "
      f"{dropped['trips'].sum() / stations['trips'].sum():.2%})")
print(dropped.groupby("reason")["trips"].agg(["size", "sum"]))
table.head()

Weggelaten stations: 70 (3,074 vertrekken = 0.01%)
                                       size   sum
reason                                           
Jersey City / Hoboken (apart systeem)    63   168
geen publiek station                      7  2906


,zone,time,trips,members,electric,temperature_c,precipitation_mm,wind_kmh,snowfall_cm,humidity_pct,split
0,0,2024-01-01 00:00:00,74,53,50,1.8,0.1,7.4,0.07,79,train
1,0,2024-01-01 01:00:00,125,95,91,1.3,0.0,8.7,0.00,81,train
2,0,2024-01-01 02:00:00,113,79,91,2.4,0.0,10.4,0.00,79,train
3,0,2024-01-01 03:00:00,52,37,40,2.7,0.0,9.2,0.00,79,train
4,0,2024-01-01 04:00:00,39,21,26,2.7,0.0,7.9,0.00,79,train


## Controles
1. Het rooster is volledig: 30 zones × 8.784 uren, geen dubbele rijen.
2. Geen ontbrekende waarden.
3. **Er gaan geen vertrekken verloren**: de som van `trips` is gelijk aan alle vertrekken van de behouden stations in 2024.
4. De testset bevat elke maand precies 7 dagen.

In [3]:
assert len(table) == cb.N_ZONES * 366 * 24
assert not table.duplicated(["zone", "time"]).any()
assert table.notna().all().all(), table.isna().sum()

kept_ids = set(public_stations["start_station_id"])
expected = station_hour.loc[station_hour["start_station_id"].isin(kept_ids) & (station_hour["time"].dt.year == cb.YEAR), "trips"].sum()
assert table["trips"].sum() == expected, (table["trips"].sum(), expected)

test_days = table.loc[table["split"] == "test", "time"].dt.normalize().drop_duplicates()
assert (test_days.dt.month.value_counts() == 7).all()
print(f"OK: {len(table):,} rijen, {table['trips'].sum():,} vertrekken, "
      f"{(table['split'] == 'test').mean():.1%} van de uren in de testset")

OK: 263,520 rijen, 44,269,890 vertrekken, 23.0% van de uren in de testset


## Opslaan
- `model_table.parquet`: de tabel voor alle modellen. Dit kleine bestand (2,3 MB) staat in Git, zodat teamgenoten de modellen kunnen hertrainen zonder 8,7 GB te downloaden.
- `zones.json`: id, naam, centrum en grootte van elke zone (gebruikt door de API en de webpagina).
- `station_zones.parquet`: welk station in welke zone ligt (voor kaarten en controle).

In [4]:
columns = ["zone", "time", "trips", "members", "electric", "temperature_c", "precipitation_mm", "wind_kmh",
           "snowfall_cm", "humidity_pct", "split"]
table = table[columns].astype({"zone": "int16", "trips": "int32", "members": "int32", "electric": "int32"})
table.to_parquet(cb.DATA_DIR / "model_table.parquet", index=False)
zones_records = zones.assign(lat=zones["lat"].round(5), lng=zones["lng"].round(5)).to_dict("records")
(cb.DATA_DIR / "zones.json").write_text(json.dumps(zones_records, indent=2, default=int), encoding="utf-8")
public_stations.to_parquet(cb.DATA_DIR / "station_zones.parquet", index=False)
print(f"model_table.parquet: {(cb.DATA_DIR / 'model_table.parquet').stat().st_size / 1e6:.1f} MB")

model_table.parquet: 2.3 MB
